# Numeric string conversion

**Purpose:** The round level fight_stats table stores all its numeric content as strings ('12 of 30', '40%', '4:32', 'Round 1'). The aim is to convert them to numeric columns so the data can feed (a) the fight dominance modifier on the competitive axis and (b) the GMM style clustering.

**Input:** the frozen fight_stats table (project scraper)

**Output:** numeric round by round table with landed and attempted split out, percentages as floats, control time in seconds, ROUND as integer.

## Section 1: Load and inspect

Load fight_stats, apply string hygiene, and print the actual columns and a sample of values. The parsers below assume specific column names and formats confirmed in the notebook 01 EDA.

In [ ]:
# BLOCK 1: Install and setup libraries

import pandas as pd
import numpy as np
import re
from datetime import datetime
from pathlib import Path

# data location (portable across Drive and a local repo checkout)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    pass  # not in Colab

DRIVE_DIR = Path('/content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/'
                 'Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs')
OUTPUT_DIR = DRIVE_DIR if DRIVE_DIR.exists() else Path('./data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using data directory: {OUTPUT_DIR}')

In [ ]:
# BLOCK 2: Load fight_stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

BASE_URL = "https://raw.githubusercontent.com/th1555/ufc-fighter-value-mapper/refs/heads/main/raw_data/"

df = pd.read_csv(BASE_URL + 'ufc_fight_stats.csv')
print(f"Loaded fight_stats: {len(df):,} rows")

# Strip all text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")

Mounted at /content/drive
Loaded fight_stats: 41,255 rows

Columns (19):
  EVENT
  BOUT
  ROUND
  FIGHTER
  KD
  SIG.STR.
  SIG.STR. %
  TOTAL STR.
  TD
  TD %
  SUB.ATT
  REV.
  CTRL
  HEAD
  BODY
  LEG
  DISTANCE
  CLINCH
  GROUND


In [ ]:
# BLOCK 3: Sample column values

print("Sample values per column:")
for c in df.columns:
    sample = df[c].dropna().astype(str).head(3).tolist()
    print(f"  {c:18s}: {sample}")

Sample values per column:
  EVENT             : ['UFC Fight Night: Allen vs. Costa', 'UFC Fight Night: Allen vs. Costa', 'UFC Fight Night: Allen vs. Costa']
  BOUT              : ['Arnold Allen vs. Melquizael Costa', 'Arnold Allen vs. Melquizael Costa', 'Arnold Allen vs. Melquizael Costa']
  ROUND             : ['Round 1', 'Round 2', 'Round 3']
  FIGHTER           : ['Arnold Allen', 'Arnold Allen', 'Arnold Allen']
  KD                : ['1.0', '0.0', '0.0']
  SIG.STR.          : ['9 of 9', '23 of 38', '32 of 52']
  SIG.STR. %        : ['100%', '60%', '61%']
  TOTAL STR.        : ['18 of 20', '28 of 44', '44 of 66']
  TD                : ['2 of 2', '0 of 0', '0 of 0']
  TD %              : ['100%', '---', '---']
  SUB.ATT           : ['0.0', '0.0', '0.0']
  REV.              : ['0.0', '0.0', '0.0']
  CTRL              : ['1:44', '0:36', '1:22']
  HEAD              : ['8 of 8', '16 of 31', '27 of 47']
  BODY              : ['0 of 0', '2 of 2', '2 of 2']
  LEG               : ['1 of 1', '

## Section 2: Identify columns by encoding type

Group the columns by which parser they need. Detect the type by inspecting the actual values rather than by column name, so the notebook adapts if the source data shifts.

In [ ]:
# BLOCK 4: Classify each column by its value encoding

# Detection heuristics:
# 'X of Y' > matches the pattern '<digits> of <digits>' 'NN%' > ends with '%'
#   'mm:ss'   > matches '<digits>:<digits>'
#   'Round N' > starts with 'Round '
# Identifier/text columns (like EVENT, BOUT, FIGHTER) are left as they are.

xy_pattern = re.compile(r'^\d+ of \d+$')
pct_pattern = re.compile(r'^\d{1,3}%$')
time_pattern = re.compile(r'^\d{1,2}:\d{2}$')
round_pattern = re.compile(r'^round \d+$', re.IGNORECASE)

def detect_encoding(series, sample_size=200):
    vals = series.dropna().astype(str).str.strip()
    if len(vals) == 0:
        return 'other'
    vals = vals.head(sample_size)
    counts = {'xy': 0, 'pct': 0, 'time': 0, 'round': 0, 'other': 0}
    for v in vals:
        if xy_pattern.match(v):
            counts['xy'] += 1
        elif pct_pattern.match(v):
            counts['pct'] += 1
        elif time_pattern.match(v):
            counts['time'] += 1
        elif round_pattern.match(v):
            counts['round'] += 1
        else:
            counts['other'] += 1
    return max(counts, key=counts.get)

encodings = {c: detect_encoding(df[c]) for c in df.columns}

print("Encoding type per column:")
for enc_type in ['xy', 'pct', 'time', 'round', 'other']:
    cols = [c for c, e in encodings.items() if e == enc_type]
    print(f"\n  {enc_type.upper()} ({len(cols)}): {cols}")

Encoding type per column:

  XY (9): ['SIG.STR.', 'TOTAL STR.', 'TD', 'HEAD', 'BODY', 'LEG', 'DISTANCE', 'CLINCH', 'GROUND']

  PCT (2): ['SIG.STR. %', 'TD %']

  TIME (1): ['CTRL']

  ROUND (1): ['ROUND']

  OTHER (6): ['EVENT', 'BOUT', 'FIGHTER', 'KD', 'SUB.ATT', 'REV.']


## Section 3: Parse 'X of Y' fields

Each 'X of Y' field becomes two integers: `<n>_landed` and `<n>_attempted`. Keeping both is deliberate for the style clustering.

In [ ]:
# BLOCK 5: Split 'X of Y' into landed and attempted

def parse_x_of_y(value):
    if pd.isna(value):
        return (np.nan, np.nan)
    s = str(value).strip()
    m = xy_pattern.match(s)
    if not m:
        return (np.nan, np.nan)
    landed, attempted = s.split(' of ')
    return (int(landed), int(attempted))

xy_cols = [c for c, e in encodings.items() if e == 'xy']

for col in xy_cols:
    parsed = df[col].apply(parse_x_of_y)
    df[f'{col}_landed'] = parsed.apply(lambda t: t[0])
    df[f'{col}_attempted'] = parsed.apply(lambda t: t[1])
    # Report null rate so any gaps are visible
    null_rate = df[f'{col}_landed'].isna().mean() * 100
    print(f"  {col:16s} > {col}_landed, {col}_attempted   (null: {null_rate:.1f}%)")

print("\nColumn Sample:")
preview_cols = []
for col in xy_cols[:2]:
    preview_cols += [col, f'{col}_landed', f'{col}_attempted']
print(df[preview_cols].head().to_string())

  SIG.STR.         > SIG.STR._landed, SIG.STR._attempted   (null: 0.1%)
  TOTAL STR.       > TOTAL STR._landed, TOTAL STR._attempted   (null: 0.1%)
  TD               > TD_landed, TD_attempted   (null: 0.1%)
  HEAD             > HEAD_landed, HEAD_attempted   (null: 0.1%)
  BODY             > BODY_landed, BODY_attempted   (null: 0.1%)
  LEG              > LEG_landed, LEG_attempted   (null: 0.1%)
  DISTANCE         > DISTANCE_landed, DISTANCE_attempted   (null: 0.1%)
  CLINCH           > CLINCH_landed, CLINCH_attempted   (null: 0.1%)
  GROUND           > GROUND_landed, GROUND_attempted   (null: 0.1%)

Column Sample:
   SIG.STR.  SIG.STR._landed  SIG.STR._attempted TOTAL STR.  TOTAL STR._landed  TOTAL STR._attempted
0    9 of 9              9.0                 9.0   18 of 20               18.0                  20.0
1  23 of 38             23.0                38.0   28 of 44               28.0                  44.0
2  32 of 52             32.0                52.0   44 of 66               4

## Section 4: Parse percentages, control time, and ROUND

Percentages become floats in [0, 1]. Control time becomes integer seconds, with the decision on nulls made clear. ROUND becomes integer.

In [ ]:
# BLOCK 6: Parse percentage fields

# 'NN%' > NN/100 as a float. Dash/null markers > NaN.
# Note: the source rounds to whole percents, so for the clustering accuracy
# computed from landed/attempted (more precise) and the parsed percents kept
# as a validation check only.

def parse_percent(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not pct_pattern.match(s):
        return np.nan  # dash or unexpected
    return int(s.rstrip('%')) / 100.0

pct_cols = [c for c, e in encodings.items() if e == 'pct']
print(f"Parsing {len(pct_cols)} percentage columns:")
for col in pct_cols:
    df[f'{col}_frac'] = df[col].apply(parse_percent)
    null_rate = df[f'{col}_frac'].isna().mean() * 100
    print(f"  {col:16s} > {col}_frac   (null: {null_rate:.1f}%)")

Parsing 2 percentage columns:
  SIG.STR. %       > SIG.STR. %_frac   (null: 0.6%)
  TD %             > TD %_frac   (null: 46.6%)


In [ ]:
# BLOCK 7: Parse control time

# Null CTRL treated as 0 seconds, NOT NaN because UFCStats records explicit
# times when control happened; a dash means 'no ground control this round'

CTRL_NULL_AS_ZERO = True

def parse_mmss(value, null_as_zero=True):
    if pd.isna(value):
        return 0 if null_as_zero else np.nan
    s = str(value).strip()
    if not time_pattern.match(s):
        # dash markers ('---', '--') or anything non time
        return 0 if null_as_zero else np.nan
    mins, secs = s.split(':')
    return int(mins) * 60 + int(secs)

time_cols = [c for c, e in encodings.items() if e == 'time']
print(f"Parsing {len(time_cols)} time columns (null_as_zero={CTRL_NULL_AS_ZERO}):")
for col in time_cols:
    df[f'{col}_secs'] = df[col].apply(lambda v: parse_mmss(v, CTRL_NULL_AS_ZERO))
    print(f"  {col:16s} > {col}_secs   (max: {df[f'{col}_secs'].max():.0f}s, mean: {df[f'{col}_secs'].mean():.0f}s)")

Parsing 1 time columns (null_as_zero=True):
  CTRL             > CTRL_secs   (max: 300s, mean: 55s)


In [ ]:
# BLOCK 8: Parse ROUND ('Round N' > integer N)

# Normalises ROUND to match the integer ROUND in fight_results

def parse_round(value):
    if pd.isna(value):
        return np.nan
    m = re.match(r'round\s+(\d+)', str(value).strip(), re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

round_cols = [c for c, e in encodings.items() if e == 'round']
for col in round_cols:
    df[f'{col}_int'] = df[col].apply(parse_round)
    n_null = df[f'{col}_int'].isna().sum()
    print(f"  {col} > {col}_int   (NaN rows: {n_null})")
    print(f"    unique values: {sorted(df[f'{col}_int'].dropna().unique())}")

  ROUND > ROUND_int   (NaN rows: 21)
    unique values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


In [ ]:
# BLOCK 9: Count columns to numeric

# Edit to  numeric; KD, SUB.ATT, and REV

COUNT_COLS = ['KD', 'SUB.ATT', 'REV.']

print("Casting count columns to numeric:")
for col in COUNT_COLS:
    df[f'{col}_num'] = pd.to_numeric(df[col], errors='coerce')
    null_rate = df[f'{col}_num'].isna().mean() * 100
    print(f"  {col:10s} -> {col}_num   (null: {null_rate:.1f}%)")

print()
print("Note: NaN here means a dash placeholder in the source.")
print("These will be treated as 0 at aggregation time (no event = zero count).")

Casting count columns to numeric:
  KD         -> KD_num   (null: 0.1%)
  SUB.ATT    -> SUB.ATT_num   (null: 0.1%)
  REV.       -> REV._num   (null: 0.1%)

Note: NaN here means a dash placeholder in the source.
These will be treated as 0 at aggregation time (no event = zero count).


## Section 5: Validate

Check the parsed columns: no strings left in numeric columns, percentages in [0, 1], and correctness from the eye test (landed never exceeds attempted, and so on).

In [ ]:
# BLOCK 10: Validate

print("VALIDATION")
print()

# landed not more than attempted
print("landed not more than attempted check:")
for col in xy_cols:
    landed = df[f'{col}_landed']
    attempted = df[f'{col}_attempted']
    # Only compare where both are not null
    mask = landed.notna() & attempted.notna()
    violations = (landed[mask] > attempted[mask]).sum()
    flag = 'OK' if violations == 0 else f'WARN: {violations} violations'
    print(f"   {col:16s}: {flag}")

# percentages in [0, 1]
print("\npercentage range check:")
for col in pct_cols:
    frac = df[f'{col}_frac'].dropna()
    out_of_range = ((frac < 0) | (frac > 1)).sum()
    flag = 'OK' if out_of_range == 0 else f'WARN: {out_of_range} out of [0,1]'
    print(f"   {col:16s}: {flag}")

# control time within max 300s
print("\ncontrol time plausibility:")
for col in time_cols:
    secs = df[f'{col}_secs']
    over_300 = (secs > 300).sum()
    print(f"   {col:16s}: max {secs.max():.0f}s, rows over 300s: {over_300}")

# no object-dtype in the new numeric columns
print("\ndtype check:")
new_cols = ([f'{c}_landed' for c in xy_cols] + [f'{c}_attempted' for c in xy_cols]
            + [f'{c}_frac' for c in pct_cols] + [f'{c}_secs' for c in time_cols]
            + [f'{c}_int' for c in round_cols]
            + [f'{c}_num' for c in COUNT_COLS])
non_numeric = [c for c in new_cols if not pd.api.types.is_numeric_dtype(df[c])]
print(f"   Non-numeric parsed columns: {non_numeric if non_numeric else 'none (all numeric, OK)'}")

VALIDATION

landed not more than attempted check:
   SIG.STR.        : OK
   TOTAL STR.      : OK
   TD              : OK
   HEAD            : OK
   BODY            : OK
   LEG             : OK
   DISTANCE        : OK
   CLINCH          : OK
   GROUND          : OK

percentage range check:
   SIG.STR. %      : OK
   TD %            : OK

control time plausibility:
   CTRL            : max 300s, rows over 300s: 0

dtype check:
   Non-numeric parsed columns: none (all numeric, OK)


## Section 6: Save

Save the parsed round table. It is the source for the dominance modifier and the GMM style clustering.

In [ ]:
# BLOCK 11: Save

# Keep the original string columns alongside the parsed ones

df.to_parquet(OUTPUT_DIR / 'fight_stats_numeric.parquet', index=False)
print(f"Saved: {OUTPUT_DIR / 'fight_stats_numeric.parquet'}")

print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)} (original + parsed)")

Saved: /content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs/fight_stats_numeric.parquet
  Rows: 41,255
  Columns: 44 (original + parsed)


## Section 7: Summary

All string-encoded round level fields are converted to numeric, and the clean parquet is saved.

**Data parsed**
- 9 'X of Y' strike and takedown fields, split into _landed and _attempted integer pairs
- 2 percentage fields, converted to _frac floats in [0, 1]
- CTRL (control time), converted to _secs integers; null treated as 0 (a genuine no-control round)
- ROUND, converted to _int integers 1 to 5; 21 NaN rows retained for downstream handling
- KD, SUB.ATT, REV., converted to _num floats; these feed both the dominance modifier and the GMM clustering

**Validated:** landed never exceeds attempted, percentages in range, control times plausible, all parsed columns numeric.

**TD % note:** the 46.6% null rate reflects rounds with zero takedown attempts (0/0 is undefined). Use TD_landed and TD_attempted directly for career accuracy rather than averaging this column.

**Outputs:** fight_stats_numeric.parquet (41,255 rows)